# Texas RRC Form R-1 Extraction Pipeline

**Extraction tiers:** Tier 1 (pdfplumber, digital) → Tier 2 (Tesseract OCR, scans) → Tier 3 (Claude API vision + retry)

**Crash-safe:** Results saved after every PDF. Restart picks up where you left off. No duplicate API calls.

### Setup
```bash
pip install pdfplumber pytesseract Pillow numpy scipy pandas requests beautifulsoup4 openpyxl PyMuPDF anthropic
```
Windows: install Tesseract from https://github.com/UB-Mannheim/tesseract/wiki

```bash
set ANTHROPIC_API_KEY=sk-ant-...
```

In [1]:
import os, re, json, base64, io, tempfile, subprocess, time, traceback
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import pdfplumber
import pytesseract
from PIL import Image
from scipy.signal import find_peaks


from dotenv import load_dotenv

# Load credentials from XX/.env — one level up from this notebook's directory
load_dotenv(dotenv_path=Path('.env'))

ANTHROPIC_API_KEY = os.getenv('ANTHROPIC_API_KEY', '').strip("'\"")

try:
    import fitz
    HAS_FITZ = True
except ImportError:
    HAS_FITZ = False
    print('WARNING: pip install PyMuPDF')

try:
    from anthropic import Anthropic
    HAS_ANTHROPIC = True
except ImportError:
    HAS_ANTHROPIC = False
    print('INFO: pip install anthropic (Tier 3 disabled)')

try:
    import camelot
    HAS_CAMELOT = True
except ImportError:
    HAS_CAMELOT = False
    print('INFO: pip install camelot-py[cv] (Tier 1b disabled)')

RAW_DATA_DIR  = Path('Raw Data/Texas RRC')
OUTPUT_DIR    = Path('data/extracted')    # raw pipeline output per facility
CLEANED_DIR   = Path('data/cleaned')     # manually curated CSVs — source of truth for analysis
LOG_DIR       = Path('logs')             # extraction logs
CHECKPOINT_DIR = LOG_DIR / 'checkpoints' # crash-safe state

for d in [OUTPUT_DIR, CLEANED_DIR, LOG_DIR, CHECKPOINT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

EXISTING_DB_PATH = 'output_cleaned_v2_1.xlsx'

# Windows Tesseract — uncomment and adjust:
pytesseract.pytesseract.tesseract_cmd = r'C:\Users\luiscarlos.gaitan\AppData\Local\Programs\Tesseract-OCR\tesseract.exe'

PRODUCT_CODES = {1:'Propane',2:'Butane',3:'Butane-Propane',4:'Motor Gasoline',5:'Kerosene',6:'Home Heating Oil',7:'Diesel Fuel',8:'Other Middle Distillates',9:'Aviation Gasoline',10:'Kerosene-Type Jet Fuel',11:'Naphtha-Type Jet Fuel',12:'Fuel Oils #4 For Utility Use',13:'Fuel Oils #5,#6 For Utility Use',14:'Fuel Oils #4 For Non-Utility Use',15:'Fuel Oils #5,#6 For Non-Utility Use',16:'Bunker C',17:'Navy Special',18:'Other Residual Fuels',19:'Petrochemical Feedstock',20:'Lubricants',21:'Special Naphthas',22:'Solvent Products',23:'Miscellaneous',24:'Crude Oil'}
DATA_COLUMNS = ['Storage Beginning of Month','Receipts','Input','Products Manufactured','Fuel Used','Deliveries','Storage End of Month']

print(f'Ready. PyMuPDF={HAS_FITZ}, Camelot={HAS_CAMELOT}, Claude={HAS_ANTHROPIC}')

Ready. PyMuPDF=True, Camelot=True, Claude=True


---
## 1. Extraction Engine

In [2]:
# ── Helpers ──

def parse_number(raw) -> float:
    if raw is None: return 0.0
    s = str(raw).strip().replace('.', '').replace(',', '').replace(' ', '').replace('|', '')
    negative = s.startswith('(') and s.endswith(')')
    s = re.sub(r'[^0-9.\-]', '', s)
    if not s or s == '-': return 0.0
    try:
        v = float(s)
        return -v if negative else v
    except:
        return 0.0

def rasterize_page(pdf_path, dpi=300):
    if HAS_FITZ:
        doc = fitz.open(pdf_path)
        zoom = dpi / 72
        pix = doc[0].get_pixmap(matrix=fitz.Matrix(zoom, zoom))
        img = Image.frombytes('RGB', [pix.width, pix.height], pix.samples)
        doc.close()
        return img
    with tempfile.TemporaryDirectory() as d:
        subprocess.run(['pdftoppm','-jpeg','-r',str(dpi),'-f','1','-l','1',pdf_path,f'{d}/p'],check=True,capture_output=True)
        return Image.open(sorted(Path(d).glob('p-*.jpg'))[0]).copy()

def image_to_base64(img, max_bytes=4_500_000):
    """Compress image to JPEG base64 under API size limit."""
    for quality in [90, 75, 60, 45]:
        buf = io.BytesIO()
        img.save(buf, format='JPEG', quality=quality)
        if buf.tell() <= max_bytes:
            return base64.b64encode(buf.getvalue()).decode('utf-8')
    scale = 0.75
    while scale > 0.3:
        resized = img.resize((int(img.width*scale), int(img.height*scale)), Image.LANCZOS)
        buf = io.BytesIO()
        resized.save(buf, format='JPEG', quality=60)
        if buf.tell() <= max_bytes:
            return base64.b64encode(buf.getvalue()).decode('utf-8')
        scale -= 0.1
    buf = io.BytesIO()
    img.resize((int(img.width*0.3), int(img.height*0.3)), Image.LANCZOS).save(buf, format='JPEG', quality=50)
    return base64.b64encode(buf.getvalue()).decode('utf-8')

def is_digital_pdf(path):
    try:
        with pdfplumber.open(path) as pdf:
            return len((pdf.pages[0].extract_text() or '').strip()) > 200
    except: return False

def extract_header_text(text):
    h = {}
    m = re.search(r'Operator[:\s]+(.+?)(?:Month|Address|$)', text, re.IGNORECASE)
    if m: h['operator'] = m.group(1).strip().rstrip('.')
    m = re.search(r'(January|February|March|April|May|June|July|August|September|October|November|December)\s+\d{4}', text, re.IGNORECASE)
    if m: h['month_str'] = m.group(0)
    m = re.search(r'(\d{2}-\d{4})', text)
    if m: h['registration_no'] = m.group(1)
    return h

def validate_balance(df):
    df = df.copy()
    df['balance_diff'] = (df['Storage Beginning of Month']+df['Receipts']+df['Products Manufactured']) - (df['Input']+df['Fuel Used']+df['Deliveries']+df['Storage End of Month'])
    df['balanced'] = df['balance_diff'].abs() < 2
    return df

def balance_stats(df):
    active = df[df[DATA_COLUMNS].sum(axis=1)>0]
    n = len(active)
    b = int(active['balanced'].sum()) if n else 0
    return {'total_active':n,'balanced_count':b,'pass_rate':round(b/n,3) if n else 1.0}

print('Helpers loaded.')

Helpers loaded.


In [3]:
# ── Tier 1: Digital PDF ──

def extract_digital(path):
    with pdfplumber.open(path) as pdf:
        text = pdf.pages[0].extract_text() or ''
        header = extract_header_text(text)
        tables = pdf.pages[0].extract_tables()
        if not tables:
            return {'header':header,'data':[],'mode':'digital','tier':3,'error':'No table','balance_stats':{},'api_usage':{'cost_usd':0,'calls':0}}
        rows = []
        for r in tables[0][1:]:
            if len(r) < 9: continue   # guard first — empty/merged rows have no r[0]
            if r[0] and 'TOTALS' in str(r[0]): continue
            c = parse_number(r[1])
            if c == 0: continue
            rows.append({'Code':int(c),'Storage Beginning of Month':parse_number(r[2]),'Receipts':parse_number(r[3]),'Input':parse_number(r[4]),'Products Manufactured':parse_number(r[5]),'Fuel Used':parse_number(r[6]),'Deliveries':parse_number(r[7]),'Storage End of Month':parse_number(r[8])})
        df = validate_balance(pd.DataFrame(rows)) if rows else pd.DataFrame()
        return {'header':header,'data':rows,'mode':'digital','tier':1,'error':None,
                'balance_stats':balance_stats(df) if not df.empty else {},'api_usage':{'cost_usd':0,'calls':0}}

print('Tier 1 loaded.')

Tier 1 loaded.


In [4]:
# ── Tier 2: Tesseract OCR ──

def detect_grid(img):
    arr = np.array(img.convert('L')); th = 100
    rk = (arr<th).sum(axis=1)

    # Try increasingly relaxed thresholds before giving up.
    # Tight thresholds work for clean scans; loose ones catch degraded ones.
    for h_frac, min_dist, min_hp, min_di, v_frac in [
        (0.15, 12, 10, 5, 0.25),   # original (strict)
        (0.10,  8,  8, 4, 0.18),   # moderate
        (0.07,  6,  6, 3, 0.12),   # loose
    ]:
        hp, _ = find_peaks(rk, height=arr.shape[1]*h_frac, distance=min_dist)
        hp = hp[(hp >= arr.shape[0]*0.15) & (hp <= arr.shape[0]*0.9)]
        if len(hp) < min_hp:
            continue
        sp = np.diff(hp); di = np.where((sp >= 18) & (sp <= 50))[0]
        if len(di) < min_di:
            continue
        thl = hp[max(0, di[0]-2):min(len(hp), di[-1]+3)].tolist()
        tr = arr[thl[0]:thl[-1], :]; ck = (tr < th).sum(axis=0)
        vp, _ = find_peaks(ck, height=len(tr)*v_frac, distance=25)
        return vp.tolist(), thl

    return None, None

def id_columns(vl):
    if len(vl)<4: return []
    gaps=[(vl[i],vl[i+1],vl[i+1]-vl[i]) for i in range(len(vl)-1)]
    wide=[(a,b,w) for a,b,w in gaps if w>100]
    if len(wide)>=8: return [(a,b) for a,b,w in wide[1:8]]
    if len(wide)==7: return [(a,b) for a,b,w in wide]
    return [(a,b) for a,b,w in gaps[-7:]] if len(gaps)>=7 else []

def find_start(hl,vl,img):
    if not vl or len(vl)<4: return min(2,len(hl)-25)
    gaps=[(vl[i],vl[i+1],vl[i+1]-vl[i],i) for i in range(len(vl)-1)]
    lh=[g for g in gaps if g[0]<img.width*0.4]
    if not lh: return min(2,len(hl)-25)
    mi=max(lh,key=lambda g:g[2])[3]
    cx1,cx2=vl[mi+1],vl[min(mi+2,len(vl)-1)]
    for i in range(min(15,len(hl)-1)):
        y1,y2=hl[i]+2,hl[i+1]-2
        if y2-y1<8: continue
        c=img.crop((cx1,y1,cx2,y2)); w,h=c.size
        if w<5 or h<5: continue
        c=c.resize((w*4,h*4),Image.LANCZOS)
        t=pytesseract.image_to_string(c,config='--psm 7 -c tessedit_char_whitelist=0123456789').strip()
        if t=='1': return i
    sp=[hl[i+1]-hl[i] for i in range(len(hl)-1)]
    for i in range(1,len(sp)):
        if sp[i]<45 and sp[i-1]>=45: return i
    return min(2,max(0,len(hl)-26))

def extract_scanned(path, dpi=200):
    try: img=rasterize_page(path,dpi)
    except Exception as e:
        return {'header':{},'data':[],'mode':'scanned_ocr','tier':3,'error':str(e),'balance_stats':{},'api_usage':{'cost_usd':0,'calls':0}}
    header=extract_header_text(pytesseract.image_to_string(img.crop((0,0,img.width,int(img.height*0.25)))))
    vl,hl=detect_grid(img)
    if not vl or not hl:
        return {'header':header,'data':[],'mode':'scanned_ocr','tier':3,'error':'No grid','balance_stats':{},'api_usage':{'cost_usd':0,'calls':0}}
    cb=id_columns(vl)
    if len(cb)<7:
        return {'header':header,'data':[],'mode':'scanned_ocr','tier':3,'error':f'{len(cb)} cols','balance_stats':{},'api_usage':{'cost_usd':0,'calls':0}}
    ds=find_start(hl,vl,img); PAD,SCALE=3,4; rows=[]
    for code in range(1,25):
        hi=ds+code-1
        if hi <0 or hi+1>=len(hl): break
        yt,yb=hl[hi]+PAD,hl[hi+1]-PAD
        if yb<=yt: continue
        rd={'Code':code}
        for ci,cn in enumerate(DATA_COLUMNS):
            if ci>=len(cb): rd[cn]=0.0; continue
            x1,x2=cb[ci]; cell=img.crop((x1+PAD,yt,x2-PAD,yb)); w,h=cell.size
            if w<5 or h<5: rd[cn]=0.0; continue
            cell=cell.resize((w*SCALE,h*SCALE),Image.LANCZOS)
            t=pytesseract.image_to_string(cell,config='--psm 7 -c tessedit_char_whitelist=0123456789,.-').strip()
            rd[cn]=parse_number(t)
        rows.append(rd)
    existing={r['Code'] for r in rows}
    for c in range(1,25):
        if c not in existing: rows.append({**{col:0.0 for col in DATA_COLUMNS},'Code':c})
    rows.sort(key=lambda r:r['Code'])
    df=validate_balance(pd.DataFrame(rows)); st=balance_stats(df)
    tier=1 if st['pass_rate']>=0.75 else 2 if st['pass_rate']>=0.25 else 3
    return {'header':header,'data':rows,'mode':'scanned_ocr','tier':tier,'error':None,'balance_stats':st,'api_usage':{'cost_usd':0,'calls':0}}

print('Tier 2 loaded.')

Tier 2 loaded.


In [5]:
# ── Tier 1b: Camelot (digital PDFs that pdfplumber misses) ──

def extract_camelot(path):
    if not HAS_CAMELOT:
        return {'header': {}, 'data': [], 'mode': 'camelot', 'tier': 3,
                'error': 'camelot not installed', 'balance_stats': {},
                'api_usage': {'cost_usd': 0, 'calls': 0}}
    try:
        with pdfplumber.open(path) as pdf:
            header = extract_header_text(pdf.pages[0].extract_text() or '')
    except Exception:
        header = {}

    for flavor in ('lattice', 'stream'):
        try:
            tables = camelot.read_pdf(str(path), flavor=flavor, pages='1')
            if not tables or tables[0].df.empty:
                continue
            df_raw = tables[0].df
            rows = []
            for _, row in df_raw.iterrows():
                c = parse_number(row.iloc[1])   # col 1 = Code
                if c < 1 or c > 24:
                    continue
                vals = [parse_number(row.iloc[2 + i]) for i in range(len(DATA_COLUMNS))]
                rows.append({'Code': int(c), **dict(zip(DATA_COLUMNS, vals))})
            if not rows:
                continue
            existing = {r['Code'] for r in rows}
            for c in range(1, 25):
                if c not in existing:
                    rows.append({**{col: 0.0 for col in DATA_COLUMNS}, 'Code': c})
            rows.sort(key=lambda r: r['Code'])
            result_df = validate_balance(pd.DataFrame(rows))
            st = balance_stats(result_df)
            tier = 1 if st['pass_rate'] >= 0.75 else 2 if st['pass_rate'] >= 0.25 else 3
            return {'header': header, 'data': rows, 'mode': 'camelot', 'tier': tier,
                    'error': None, 'balance_stats': st,
                    'api_usage': {'cost_usd': 0, 'calls': 0}}
        except Exception:
            continue

    return {'header': header, 'data': [], 'mode': 'camelot', 'tier': 3,
            'error': 'No table found', 'balance_stats': {},
            'api_usage': {'cost_usd': 0, 'calls': 0}}

print('Tier 1b (camelot) loaded.')

Tier 1b (camelot) loaded.


In [6]:
# ── Tier 3: Claude API with retry ──

CLAUDE_PROMPT = '''Extract the material balance table from this Texas RRC Form R-1.

The table has EXACTLY these 24 rows in this order:
1=Propane, 2=Butane, 3=Butane-Propane, 4=Motor Gasoline, 5=Kerosene,
6=Home Heating Oil, 7=Diesel Fuel, 8=Other Middle Distillates, 9=Aviation Gasoline,
10=Kerosene-Type Jet Fuel, 11=Naphtha-Type Jet Fuel, 12=Fuel Oils #4 Utility,
13=Fuel Oils #5,#6 Utility, 14=Fuel Oils #4 Non-Utility, 15=Fuel Oils #5,#6 Non-Utility,
16=Bunker C, 17=Navy Special, 18=Other Residual Fuels, 19=Petrochemical Feedstock,
20=Lubricants, 21=Special Naphthas, 22=Solvent Products, 23=Miscellaneous, 24=Crude Oil

Columns: Storage Beginning of Month, Receipts, Input Runs to Stills and/or Blends,
Products Manufactured, Fuel Used, Deliveries, Storage End of Month.

Return ONLY JSON — no markdown, no backticks:
{"operator":"...","month":"...","registration_no":"XX-XXXX","refinery_name":"...",
"rows":[{"code":1,"storage_bom":0,"receipts":0,"input":0,"products_mfg":0,"fuel_used":0,"deliveries":0,"storage_eom":0}]}

Rules:
- Use 0 for blank/dash cells. Integers only, no commas.
- EVERY row must match the product name printed on that line.
- Balance: storage_bom + receipts + products_mfg = input + fuel_used + deliveries + storage_eom
- If a number seems wrong, re-read that row carefully before committing.'''

# Errors worth retrying (transient); all others bubble up immediately.
_RETRYABLE = ('RateLimitError', 'APITimeoutError', 'APIConnectionError', 'InternalServerError')

def _call_claude(b64, prompt, model, max_retries=3):
    """Single Claude API call with exponential backoff for transient errors.
    Returns (message, error_string).
    """
    last_err = None
    for attempt in range(max_retries):
        try:
            msg = Anthropic().messages.create(model=model, max_tokens=4096, messages=[{
                'role': 'user', 'content': [
                    {'type': 'image', 'source': {'type': 'base64', 'media_type': 'image/jpeg', 'data': b64}},
                    {'type': 'text', 'text': prompt}]}])
            return msg, None
        except Exception as e:
            last_err = e
            err_type = type(e).__name__
            is_retryable = any(r in err_type for r in _RETRYABLE)
            if is_retryable and attempt < max_retries - 1:
                wait = 2 ** (attempt + 1)  # 2s, 4s, 8s
                print(f'    Claude {err_type} (attempt {attempt+1}/{max_retries}), retrying in {wait}s...')
                time.sleep(wait)
            else:
                break
    return None, f'{type(last_err).__name__}: {last_err}'


def _parse_claude_response(msg):
    """Parse Claude message into header + rows. Returns (header, rows, error)."""
    txt = re.sub(r'^```(?:json)?\s*','', msg.content[0].text.strip())
    txt = re.sub(r'\s*```$','', txt)
    try:
        p = json.loads(txt)
    except json.JSONDecodeError as e:
        return {}, [], f'JSON parse: {e}'
    header = {'operator':p.get('operator',''),'month_str':p.get('month',''),
              'registration_no':p.get('registration_no',''),'refinery_name':p.get('refinery_name','')}
    rows = []
    for r in p.get('rows',[]):
        c = int(r.get('code',0))
        if c<1 or c>24: continue
        rows.append({'Code':c,'Storage Beginning of Month':float(r.get('storage_bom',0)),
            'Receipts':float(r.get('receipts',0)),'Input':float(r.get('input',0)),
            'Products Manufactured':float(r.get('products_mfg',0)),'Fuel Used':float(r.get('fuel_used',0)),
            'Deliveries':float(r.get('deliveries',0)),'Storage End of Month':float(r.get('storage_eom',0))})
    # Fill missing codes
    existing = {r['Code'] for r in rows}
    for c in range(1,25):
        if c not in existing: rows.append({**{col:0.0 for col in DATA_COLUMNS},'Code':c})
    rows.sort(key=lambda r:r['Code'])
    return header, rows, None


def _build_carry_prompt(prev_end_context):
    """
    Build the carry-forward validation block appended to CLAUDE_PROMPT when
    the previous month's Storage End of Month values are known.

    prev_end_context : dict {code (int): storage_eom (float)}
    """
    active_lines = '\n'.join(
        f'  Code {c} ({PRODUCT_CODES.get(c, "?")}): {int(v)}'
        for c, v in sorted(prev_end_context.items())
        if v != 0   # skip inactive products to keep the prompt concise
    )
    if not active_lines:
        return ''
    return (
        '\n\nCARRY-FORWARD VALIDATION (mandatory):\n'
        'The previous month\'s Storage End of Month values are:\n'
        f'{active_lines}\n'
        'Your extracted Storage Beginning of Month for each code listed above '
        'MUST equal the value shown (within 2 barrels). '
        'If your reading differs, re-read that row — the number may be in the '
        'wrong product row on the form.'
    )



# ── Rotation cache: maps refinery_id → the image rotation (0/90/-90) that worked ──
# Stored in output/checkpoints/rotation_cache.json so it persists across sessions.
ROTATION_CACHE_PATH = CHECKPOINT_DIR / 'rotation_cache.json'

def load_rotation_cache():
    """Return the {refinery_id: rotation} dict from disk, or {} if not found."""
    if ROTATION_CACHE_PATH.exists():
        try:
            return json.loads(ROTATION_CACHE_PATH.read_text())
        except Exception:
            return {}
    return {}

def save_rotation_cache(cache):
    """Write the rotation cache to disk immediately (so a crash doesn't lose it)."""
    ROTATION_CACHE_PATH.write_text(json.dumps(cache, indent=2))

# Image orientations to try when the default view returns empty/invalid JSON.
# The rotation cache (rotation_cache.json) remembers which worked per refinery
# so subsequent months skip straight to the known-good orientation.
_RETRY_ROTATIONS = [0, 90, -90]


def extract_claude_with_retry(path, model='claude-sonnet-4-6',
                               prev_end_context=None, refinery_id=None):
    """Tier 3: Claude vision with rotation detection, carry context, and balance retry.

    prev_end_context : dict {code: storage_eom} from the previous balanced month.
        Appended to the prompt so Claude self-validates row placement.
    refinery_id : str  Used to look up / update the rotation cache so the
        correct image orientation is tried first for known refineries.
    """
    if not HAS_ANTHROPIC:
        return {'header':{},'data':[],'mode':'claude_vision','tier':3,
                'error':'anthropic not installed','balance_stats':{},'api_usage':{'cost_usd':0,'calls':0}}

    # Build rotation order: cached orientation first, then fallbacks
    cache     = load_rotation_cache()
    cached    = cache.get(refinery_id) if refinery_id else None
    rotations = ([cached] + [r for r in _RETRY_ROTATIONS if r != cached]
                 if cached is not None else _RETRY_ROTATIONS)

    # Build prompt — carry-forward block when previous month is known
    prompt = CLAUDE_PROMPT
    if prev_end_context:
        prompt += _build_carry_prompt(prev_end_context)

    # ── Pass 1: try each orientation until Claude returns parseable JSON ──
    header, rows             = {}, []
    total_in = total_out     = 0
    api_calls                = 0
    b64                      = None   # reused in Pass 2 (correctly-oriented image)

    for rotation in rotations:
        try:
            img = rasterize_page(path, 300)
            if rotation != 0:
                img = img.rotate(rotation, expand=True)
            b64 = image_to_base64(img)
        except Exception as e:
            return {'header':{},'data':[],'mode':'claude_vision','tier':3,
                    'error':f'Image: {e}','balance_stats':{},'api_usage':{'cost_usd':0,'calls':0}}

        msg1, err1 = _call_claude(b64, prompt, model)
        if err1:
            # API errors (rate-limit, auth …) are not orientation-dependent — stop immediately
            return {'header':{},'data':[],'mode':'claude_vision','tier':3,
                    'error':err1,'balance_stats':{},'api_usage':{'cost_usd':0,'calls':api_calls}}

        api_calls += 1
        total_in  += msg1.usage.input_tokens
        total_out += msg1.usage.output_tokens

        raw = msg1.content[0].text.strip()
        if not raw:
            print(f'  [empty at {rotation:+d}°]', end=' ', flush=True)
            continue   # try next orientation

        header, rows, parse_err = _parse_claude_response(msg1)
        if parse_err:
            print(f'  [parse fail at {rotation:+d}°] {repr(raw[:80])}', flush=True)
            continue   # try next orientation

        # Good parse — update cache if this orientation is new for this refinery
        if refinery_id and cache.get(refinery_id) != rotation:
            cache[refinery_id] = rotation
            save_rotation_cache(cache)
        break   # proceed to Pass 2

    else:
        # All orientations exhausted without a parseable response
        cost = (total_in * 3 + total_out * 15) / 1_000_000
        return {'header':{},'data':[],'mode':'claude_vision','tier':3,
                'error':'all rotations failed (empty/invalid JSON)','balance_stats':{},
                'api_usage':{'cost_usd':round(cost,4),'calls':api_calls,
                             'input_tokens':total_in,'output_tokens':total_out}}

    # ── Check within-month balance ──
    df = validate_balance(pd.DataFrame(rows))
    st = balance_stats(df)
    active_fails = df[(df[DATA_COLUMNS].sum(axis=1)>0) & (~df['balanced'])]

    if active_fails.empty:
        cost = (total_in*3 + total_out*15)/1_000_000
        return {'header':header,'data':rows,'mode':'claude_vision',
                'tier':1 if st['pass_rate']>=0.75 else 2,'error':None,'balance_stats':st,
                'api_usage':{'cost_usd':round(cost,4),'calls':api_calls,
                             'input_tokens':total_in,'output_tokens':total_out}}
    
    # ── Pass 2: ask Claude to re-read only the unbalanced rows ──
    # b64 already holds the correctly-oriented image from Pass 1 — no re-rotation needed.
    fail_info = [{'code':int(r['Code']),'product':PRODUCT_CODES.get(int(r['Code'])),'values':{c:r[c] for c in DATA_COLUMNS},'diff':r['balance_diff']} for _,r in active_fails.iterrows()]
    retry_prompt = f'''These rows failed the balance check (storage_bom + receipts + products_mfg should equal input + fuel_used + deliveries + storage_eom):\n\n{json.dumps(fail_info, indent=2)}\n\nRe-read ONLY these rows from the image, matching numbers to the correct product name.\nReturn ONLY a JSON array:\n[{{"code":1,"storage_bom":0,"receipts":0,"input":0,"products_mfg":0,"fuel_used":0,"deliveries":0,"storage_eom":0}}]'''
    
    msg2, err2 = _call_claude(b64, retry_prompt, model)
    if err2:
        cost = (total_in*3 + total_out*15)/1_000_000
        return {'header':header,'data':rows,'mode':'claude_vision',
                'tier':1 if st['pass_rate']>=0.75 else 2,'error':None,'balance_stats':st,
                'api_usage':{'cost_usd':round(cost,4),'calls':api_calls,'retry_error':err2,
                             'input_tokens':total_in,'output_tokens':total_out}}

    api_calls += 1
    total_in  += msg2.usage.input_tokens
    total_out += msg2.usage.output_tokens

    # Parse and apply corrections
    txt2 = re.sub(r'^```(?:json)?\s*','', msg2.content[0].text.strip())
    txt2 = re.sub(r'\s*```$','', txt2)
    try:
        corrections = json.loads(txt2)
        corrected = {int(r['code']):r for r in corrections}
        for row in rows:
            if row['Code'] in corrected:
                c = corrected[row['Code']]
                for k,v in [('storage_bom','Storage Beginning of Month'),('receipts','Receipts'),('input','Input'),('products_mfg','Products Manufactured'),('fuel_used','Fuel Used'),('deliveries','Deliveries'),('storage_eom','Storage End of Month')]:
                    row[v] = float(c.get(k, 0))
    except Exception as _retry_err:
        print(f'    Warning: retry-correction parse failed ({type(_retry_err).__name__}: {_retry_err}); keeping Pass-1 values.')
    
    df2 = validate_balance(pd.DataFrame(rows))
    st2  = balance_stats(df2)
    cost = (total_in*3 + total_out*15)/1_000_000

    return {'header':header,'data':rows,'mode':'claude_vision',
            'tier':1 if st2['pass_rate']>=0.75 else 2,'error':None,'balance_stats':st2,
            'api_usage':{'cost_usd':round(cost,4),'calls':api_calls,
                         'retried_rows':len(active_fails),
                         'input_tokens':total_in,'output_tokens':total_out}}

print('Tier 3 loaded.')

Tier 3 loaded.


In [7]:
# ── Unified extractor ──

def extract_r1(path, use_claude=True, claude_threshold=0.62,
               prev_end_context=None, refinery_id=None):
    """Unified extractor (Tier 1 pdfplumber → Tier 1b camelot → Tier 2 Tesseract → Tier 3 Claude).

    prev_end_context : dict {code: storage_eom} from the previous balanced month,
        forwarded to extract_claude_with_retry for carry-forward validation.
    refinery_id : str  Forwarded to extract_claude_with_retry for rotation caching.
    """
    if is_digital_pdf(path):
        r = extract_digital(path)
        if r['data']: return r
    r_cam = extract_camelot(path)
    if r_cam.get('data') and r_cam.get('balance_stats', {}).get('pass_rate', 0) >= claude_threshold:
        return r_cam
    r = extract_scanned(path)
    if r.get('data') and r.get('balance_stats',{}).get('pass_rate',0) >= claude_threshold:
        return r
    t2 = r
    cr = None
    if use_claude and HAS_ANTHROPIC:
        print('calling Claude - Gonna cost some bucks')
        cr = extract_claude_with_retry(path, prev_end_context=prev_end_context,
                                             refinery_id=refinery_id)
        if not cr.get('error'): return cr
    # Only fall back to low-quality OCR when Claude was never attempted
    # (no API key / anthropic not installed). If Claude was tried and failed,
    # returning garbage OCR data is worse than marking the file as failed.
    if t2.get('data') and not (use_claude and HAS_ANTHROPIC):
        t2['note'] = 'Low confidence'
        return t2
    # Propagate Claude's real error and any partial cost so the batch log is informative.
    claude_err = cr.get('error', 'claude_not_called') if cr else 'claude_not_called'
    claude_api = cr.get('api_usage', {'cost_usd': 0, 'calls': 0}) if cr else {'cost_usd': 0, 'calls': 0}
    return {'header': {}, 'data': [], 'mode': 'failed', 'tier': 3,
            'error': f'All failed | Claude: {claude_err}',
            'balance_stats': {}, 'api_usage': claude_api}

print('Unified extractor ready.')

Unified extractor ready.


---
## 2. Crash-Safe Batch Extraction

Saves results after EVERY PDF. If the loop crashes, restart and it picks up where it left off.
No duplicate API calls.

In [8]:
def load_checkpoint():
    data_path = CHECKPOINT_DIR / 'extracted_rows.csv'
    log_path = CHECKPOINT_DIR / 'extraction_log.csv'
    
    try:
        data_df = pd.read_csv(data_path, dtype={'id': str}) if data_path.exists() and data_path.stat().st_size > 0 else pd.DataFrame()
    except Exception:
        data_df = pd.DataFrame()
    
    try:
        log_df = pd.read_csv(log_path, dtype={'id': str}) if log_path.exists() and log_path.stat().st_size > 0 else pd.DataFrame()
    except Exception:
        log_df = pd.DataFrame()
    
    done = set(log_df['file'].tolist()) if not log_df.empty and 'file' in log_df.columns else set()
    return data_df, log_df, done


def _validate_api_key():
    """Fast-fail check before a batch run. Returns (ok, error_message)."""
    if not HAS_ANTHROPIC:
        return False, 'anthropic package not installed'
    key = os.environ.get('ANTHROPIC_API_KEY', '').strip()
    if not key:
        return False, 'ANTHROPIC_API_KEY is not set — run Cell 1 and check your .env file'
    if not key.startswith('sk-ant-'):
        return False, f'ANTHROPIC_API_KEY looks wrong (starts with {key[:8]!r})'
    return True, None





def _parse_pdf_yearmo(fp):
    """Extract (year, month) from filename like '2022_3___01-0692_....pdf'. Returns (None,None) on failure."""
    try:
        parts = Path(fp).stem.split('___')
        yr, mo = parts[0].split('_')[:2]
        return int(yr), int(mo)
    except Exception:
        return None, None


def batch_extract(data_dir=RAW_DATA_DIR, facility_filter=None, use_claude=True,
                  resume=True, delay_between_api=1.0,
                  date_from='2020-01', date_to=None):
    """Run extraction pipeline.

    date_from / date_to : 'YYYY-MM' strings (inclusive). PDFs outside this
        range are skipped at zero cost before any API call is made.
        Default date_from='2020-01' focuses on recent data.
        Pass date_from=None to process all available years.
    """
    # ── Validate API key upfront so we don't silently fail on every PDF ──
    if use_claude:
        ok, key_err = _validate_api_key()
        if not ok:
            print(f'ERROR: {key_err}')
            print('Set ANTHROPIC_API_KEY or pass use_claude=False to run Tier 1/2 only.')
            return pd.DataFrame(), pd.DataFrame()

    # Parse date bounds once
    from_ym = tuple(map(int, date_from.split('-'))) if date_from else None
    to_ym   = tuple(map(int, date_to.split('-')))   if date_to   else None

    pdfs = sorted(data_dir.rglob('*.pdf'))
    if facility_filter:
        pdfs = [p for p in pdfs if any(f in str(p) for f in facility_filter)]

    # Apply date filter purely on filename — no file I/O cost
    if from_ym or to_ym:
        filtered = []
        for p in pdfs:
            yr, mo = _parse_pdf_yearmo(p)
            if yr is None:
                filtered.append(p)   # keep unparseable filenames (will fail later with clear error)
                continue
            if from_ym and (yr, mo) < from_ym:
                continue
            if to_ym and (yr, mo) > to_ym:
                continue
            filtered.append(p)
        skipped = len(pdfs) - len(filtered)
        if skipped:
            print(f'Date filter: skipping {skipped} PDFs outside {date_from or "start"} – {date_to or "now"}')
        pdfs = filtered

    # Load checkpoint if resuming
    if resume:
        prev_data, prev_log, done = load_checkpoint()
        all_rows = prev_data.to_dict('records') if not prev_data.empty else []
        all_log = prev_log.to_dict('records') if not prev_log.empty else []
        n_skip = sum(1 for p in pdfs if str(p) in done)
        print(f'{len(pdfs)} PDFs total, {n_skip} already done, {len(pdfs)-n_skip} remaining.')
    else:
        all_rows, all_log, done = [], [], set()
        for f in CHECKPOINT_DIR.glob('*.csv'): f.unlink()
        print(f'{len(pdfs)} PDFs. Starting fresh.')
    
    def save_progress():
        pd.DataFrame(all_rows).to_csv(CHECKPOINT_DIR / 'extracted_rows.csv', index=False)
        pd.DataFrame(all_log).to_csv(CHECKPOINT_DIR / 'extraction_log.csv', index=False)
    
    processed, total_cost = 0, 0.0

    # carry_context[fid] = {code: storage_eom} from the last fully-balanced month.
    # Only updated when ALL active rows pass the balance check AND the activity
    # profile is consistent — so we never propagate a corrupt anchor forward.
    carry_context = {}

    for i, fp in enumerate(pdfs):
        fp_str = str(fp)
        if resume and fp_str in done:
            continue
        
        # Parse filename
        parts = fp.stem.split('___')
        if len(parts) < 2:
            all_log.append({'file': fp_str, 'id': '', 'date': '', 'mode': 'failed', 'tier': 3,
                            'error': 'bad filename', 'claude_error': '',
                            'active': 0, 'balanced': 0, 'pass_rate': 0, 'cost': 0, 'calls': 0, 'retried_rows': 0})
            continue
        
        fid = str(parts[1].split('_')[0])
        try:
            yr, mo = parts[0].split('_')[:2]; yr, mo = int(yr), int(mo)
            date = datetime(yr, mo+1, 1) - pd.Timedelta(days=1) if mo < 12 else datetime(yr, 12, 31)
            date_str = date.strftime('%Y-%m-%d')
        except:
            date_str = ''

        # Extract — pass previous month's carry context to Claude when available
        ctx = carry_context.get(fid)   # None for first month of a refinery
        try:
            r = extract_r1(fp_str, use_claude=use_claude,
                           prev_end_context=ctx, refinery_id=fid)
        except Exception as e:
            print(f'\n{"!"*60}')
            print(f'ERROR on file {i+1}/{len(pdfs)}: {fp.name}')
            print(f'{type(e).__name__}: {e}')
            print(f'Saving progress ({processed} PDFs processed, ${total_cost:.2f} spent)...')
            save_progress()
            print(f'Saved to {CHECKPOINT_DIR}')
            print(f'Fix the issue and re-run batch_extract() to resume.')
            print(f'{"!"*60}')
            raise
        
        # Accumulate results in memory
        api = r.get('api_usage', {})
        for row in r.get('data', []):
            row.update({'id': fid, 'date': date_str,
                        'extraction_mode': r['mode'], 'extraction_tier': r.get('tier', 3)})
            all_rows.append(row)
        
        raw_err = r.get('error') or ''
        # Split "All failed | Claude: <detail>" into separate columns for readability.
        if '| Claude:' in (raw_err or ''):
            top_err, claude_detail = raw_err.split('| Claude:', 1)
        else:
            top_err, claude_detail = raw_err, ''

        st = r.get('balance_stats', {})
        all_log.append({
            'file': fp_str, 'id': fid, 'date': date_str, 'mode': r['mode'], 'tier': r.get('tier', 3),
            'error': top_err.strip(), 'claude_error': claude_detail.strip(),
            'active': st.get('total_active', 0),
            'balanced': st.get('balanced_count', 0),
            'pass_rate': st.get('pass_rate', 0),
            'cost': api.get('cost_usd', 0), 'calls': api.get('calls', 0),
            'retried_rows': api.get('retried_rows', 0)})

        # Update carry context only when this month is fully balanced (cascade guard):
        # An unbalanced or failed month must never become the anchor for the next month.
        if r.get('data') and r.get('mode') != 'failed' and st.get('pass_rate', 0) >= 1.0:
            carry_context[fid] = {int(row['Code']): row.get('Storage End of Month', 0)
                                  for row in r['data']}

        processed += 1
        total_cost += api.get('cost_usd', 0)
        
        if processed % 25 == 0:
            print(f'  [{processed} done] last={r["mode"]} tier={r.get("tier")} cost=${total_cost:.2f}')
        
        if api.get('calls', 0) > 0 and delay_between_api > 0:
            time.sleep(delay_between_api)
    
    # ── All done: save final results ──
    save_progress()
    
    rdf = pd.DataFrame(all_rows)
    ldf = pd.DataFrame(all_log)
    
    print(f'\n{"="*60}')
    print(f'BATCH COMPLETE: {processed} processed, ${total_cost:.2f} API cost')
    if not ldf.empty:
        print(f'Tiers: {dict(ldf["tier"].value_counts().sort_index())}')
        print(f'Modes: {dict(ldf["mode"].value_counts())}')
        failed = ldf[ldf['mode'] == 'failed']
        if not failed.empty:
            print(f'Failed ({len(failed)}):')
            for _, row in failed.iterrows():
                print(f'  {Path(row["file"]).name} → {row.get("claude_error") or row.get("error")}')
    print(f'Saved to {CHECKPOINT_DIR}')
    
    return rdf, ldf

In [9]:
# ── Retry-only pass: re-run every mode='failed' PDF without touching successes ──

def _safe_csv_write(df, path):
    """Write a CSV, with a helpful message if the file is locked (e.g. open in Excel)."""
    try:
        df.to_csv(path, index=False)
        return True
    except PermissionError:
        print(f'\n  PermissionError: cannot write {path}')
        print('  → Close the file in Excel (or any other program) and re-run this cell.')
        return False


def retry_failed(log_csv, data_csv=None, facility_filter=None,
                 model='claude-sonnet-4-6', delay=2.0,
                 date_from='2020-01', date_to=None,
                 upgrade_ocr=False, ocr_pass_threshold=0.6):
    """Re-attempt extraction for every row where mode='failed' in log_csv.

    Parameters
    ----------
    log_csv             : path to the extraction log CSV
    data_csv            : path to the extracted-data CSV to patch in-place (optional)
    facility_filter     : list of id strings to limit retries, e.g. ['01-0692']
    model               : Claude model name
    delay               : seconds to wait between successful API calls
    date_from           : 'YYYY-MM' — skip PDFs before this month (default '2020-01')
    date_to             : 'YYYY-MM' — skip PDFs after this month (default None = no limit)
    upgrade_ocr         : when True, also re-run mode='scanned_ocr' rows whose
                          pass_rate is below ocr_pass_threshold through Claude.
                          Rows already extracted by claude_vision are never touched.
    ocr_pass_threshold  : pass_rate cut-off used when upgrade_ocr=True (default 0.6)
    """
    ok, key_err = _validate_api_key()
    if not ok:
        print(f'ERROR: {key_err}'); return None, None

    log_df = pd.read_csv(log_csv, dtype={'id': str})
    mask = log_df['mode'] == 'failed'
    if upgrade_ocr:
        # Also target low-quality OCR rows; never re-process already-successful Claude extractions
        low_ocr = (log_df['mode'] == 'scanned_ocr') & (log_df['pass_rate'].fillna(0) < ocr_pass_threshold)
        mask = mask | low_ocr
    if facility_filter:
        mask &= log_df['id'].isin(facility_filter)

    # Apply date filter on the log (file path encodes year/month)
    from_ym = tuple(map(int, date_from.split('-'))) if date_from else None
    to_ym   = tuple(map(int, date_to.split('-')))   if date_to   else None
    if from_ym or to_ym:
        def _in_range(fp):
            yr, mo = _parse_pdf_yearmo(fp)
            if yr is None: return True
            if from_ym and (yr, mo) < from_ym: return False
            if to_ym   and (yr, mo) > to_ym:   return False
            return True
        mask &= log_df['file'].apply(_in_range)

    targets = log_df[mask].copy()

    if targets.empty:
        print('No failed rows to retry.'); return log_df, None

    print(f'Retrying {len(targets)} failed PDFs...')

    data_df = pd.read_csv(data_csv, dtype={'id': str}) if data_csv and Path(data_csv).exists() else pd.DataFrame()
    new_rows, updated_log = [], []
    total_cost = 0.0  # cumulative API cost across the whole retry run

    for _, row in targets.iterrows():
        fp  = row['file']
        fid = str(row['id'])

        if not Path(fp).exists():
            print(f'  SKIP (not found): {fp}')
            updated_log.append(row.to_dict())
            continue

        print(f'  {Path(fp).name}', end=' ... ', flush=True)

        # extract_claude_with_retry already handles rotation detection, caching, and
        # Pass-2 balance retry — no duplication needed here.
        r_dict    = extract_claude_with_retry(fp, model=model, refinery_id=fid)
        api       = r_dict.get('api_usage', {})
        file_cost = api.get('cost_usd', 0)
        total_cost += file_cost

        if r_dict.get('error') or not r_dict.get('data'):
            err = r_dict.get('error', 'unknown error')
            print(f'FAIL  cost=${file_cost:.4f}  [session total ${total_cost:.4f}]  ({err})')
            r = row.to_dict()
            r['claude_error'] = err
            r['cost'] = file_cost
            updated_log.append(r)
            time.sleep(delay)
            continue

        # Success — build output rows (validate_balance adds balance_diff & balanced)
        st   = r_dict.get('balance_stats', {})
        tier = 1 if st.get('pass_rate', 0) >= 0.75 else 2 if st.get('pass_rate', 0) >= 0.25 else 3
        print(f'OK  pass_rate={st.get("pass_rate", 0):.0%}  cost=${file_cost:.4f}  [session total ${total_cost:.4f}]')

        date_str = row['date']
        df = validate_balance(pd.DataFrame(r_dict['data']))
        for rw in df.to_dict('records'):
            rw['material'] = PRODUCT_CODES.get(int(rw['Code']), '')
            rw.update({'id': fid, 'date': date_str,
                       'extraction_mode': 'claude_vision', 'extraction_tier': tier})
            new_rows.append(rw)

        r = row.to_dict()
        r.update({'mode': 'claude_vision', 'tier': tier, 'error': '',
                  'claude_error': '', 'active': st.get('total_active', 0),
                  'balanced': st.get('balanced_count', 0), 'pass_rate': st.get('pass_rate', 0),
                  'cost': file_cost, 'calls': api.get('calls', 1),
                  'retried_rows': api.get('retried_rows', 0)})
        updated_log.append(r)
        time.sleep(delay)

    # ── Merge results back into the original log ──
    result_log = log_df.copy()
    update_map = {r['file']: r for r in updated_log}
    for col in ['claude_error']:
        if col not in result_log.columns:
            result_log[col] = ''
    for idx, row in result_log.iterrows():
        if row['file'] in update_map:
            for k, v in update_map[row['file']].items():
                if k in result_log.columns:
                    result_log.at[idx, k] = v

    # ── Patch data CSV ──
    if data_csv and new_rows:
        new_df = pd.DataFrame(new_rows)
        if not data_df.empty:
            keep = ~((data_df['id'].isin(targets['id'])) &
                     (data_df['date'].isin(targets['date'])))
            data_df = pd.concat([data_df[keep], new_df], ignore_index=True)
        else:
            data_df = new_df
        if _safe_csv_write(data_df, data_csv):
            print(f'Data saved → {data_csv}')

    if _safe_csv_write(result_log, log_csv):
        print(f'Log  saved → {log_csv}')

    n_ok = sum(1 for r in updated_log if r.get('mode') == 'claude_vision')
    n_fail = len(targets) - n_ok
    print(f'\n{"="*55}')
    print(f'Retry complete: {n_ok} recovered, {n_fail} still failed')
    print(f'Session API cost: ${total_cost:.4f}')
    print(f'{"="*55}')
    return result_log, data_df

print('retry_failed() ready.')

retry_failed() ready.


In [10]:
# ── Automated Carry Repair ───────────────────────────────────────────────────
# Detects and auto-fixes months where extracted numbers are in the wrong
# product row, using three complementary detection layers:
#
#   Layer 1 — Carry-based pairwise swap
#             Uses the previous balanced month's storage_eom as "expected
#             beginning".  If two rows are transposed the swap is detectable
#             without any API call.
#
#   Layer 2 — Activity-profile anomaly swap
#             A refinery's product mix is stable.  If Motor Gasoline has been
#             active every month for three years and it suddenly shows zeros
#             while a historically-inactive product shows large values, the
#             numbers are almost certainly in the wrong row — even if both
#             rows individually pass the within-month balance check.
#
#   Layer 3 — Block shift  (±1 to ±5 positions)
#             Catches systematic row-offset errors from OCR grid misalignment.
#
# Cascade protection: a month is only trusted as a carry anchor if
#   (a) every row passed the within-month balance check, AND
#   (b) its own activity pattern is consistent with its historical profile.

CARRY_REPAIR_TOL   = 2      # barrels — same tolerance as existing balance check
INACTIVE_THRESHOLD = 0.05   # product active <5% of history → "typically inactive"
ACTIVE_THRESHOLD   = 0.30   # product active >30% of history → "regularly active"
PROFILE_WINDOW     = 24     # look-back window in months for the activity profile


def _fully_balanced_months(fac_df):
    """Set of dates where every row passed the within-month balance check."""
    return set(
        fac_df.groupby('date')['balanced']
        .all()
        .pipe(lambda s: s[s].index)
    )


def _activity_profile(history_df):
    """Dict {code (int): activity_rate (float)} for a given history slice."""
    if history_df.empty:
        return {}
    result = {}
    for code, grp in history_df.groupby('Code'):
        result[int(code)] = float((grp[DATA_COLUMNS].sum(axis=1) > 0).mean())
    return result


def _is_anchor_trustworthy(cand_rows, anc_profile):
    """Return False if a 'never-active' product suddenly appears in this month."""
    for _, row in cand_rows.iterrows():
        code = int(row['Code'])
        is_active = float(row[DATA_COLUMNS].sum()) > 0
        if is_active and anc_profile.get(code, 1.0) < INACTIVE_THRESHOLD:
            return False
    return True


def _carry_bad_codes(month_idx, anchor_idx, tol):
    """Codes where |this_month.storage_bom - anchor.storage_eom| > tol."""
    bad = []
    for code in range(1, 25):
        if code not in month_idx.index or code not in anchor_idx.index:
            continue
        actual   = float(month_idx.loc[code, 'Storage Beginning of Month'])
        expected = float(anchor_idx.loc[code, 'Storage End of Month'])
        if abs(actual - expected) > tol:
            bad.append(code)
    return bad


def _apply_row_swaps(df_idx, pairs):
    """Return copy of df_idx with DATA_COLUMNS swapped for each (A, B) pair."""
    out = df_idx.copy()
    for a, b in pairs:
        if a in out.index and b in out.index:
            tmp = out.loc[a, DATA_COLUMNS].copy()
            out.loc[a, DATA_COLUMNS] = out.loc[b, DATA_COLUMNS].values
            out.loc[b, DATA_COLUMNS] = tmp.values
    return out


def _find_pairwise_swaps(month_idx, anchor_idx, bad_codes, tol):
    """Return list of (A, B) code pairs whose swap restores carry consistency."""
    swaps, remaining = [], set(bad_codes)
    for i, a in enumerate(bad_codes):
        if a not in remaining:
            continue
        for b in bad_codes[i + 1:]:
            if b not in remaining:
                continue
            actual_a  = float(month_idx.loc[a,  'Storage Beginning of Month']) if a  in month_idx.index  else 0.0
            actual_b  = float(month_idx.loc[b,  'Storage Beginning of Month']) if b  in month_idx.index  else 0.0
            expect_a  = float(anchor_idx.loc[a, 'Storage End of Month'])        if a  in anchor_idx.index else 0.0
            expect_b  = float(anchor_idx.loc[b, 'Storage End of Month'])        if b  in anchor_idx.index else 0.0
            if abs(actual_a - expect_b) <= tol and abs(actual_b - expect_a) <= tol:
                swaps.append((a, b))
                remaining.discard(a)
                remaining.discard(b)
    return swaps


def _find_block_shift(month_idx, anchor_idx, tol, max_shift=5):
    """
    Try circular shifts ±1..max_shift of all 24 rows.
    Return (shift_n, shifted_idx) if a shift resolves ALL carry failures,
    else (None, None).
    """
    all_codes = list(range(1, 25))

    def apply_shift(n):
        shifted = month_idx.copy()
        for i, src in enumerate(all_codes):
            dst = all_codes[(i - n) % 24]
            if src in month_idx.index and dst in shifted.index:
                shifted.loc[dst, DATA_COLUMNS] = month_idx.loc[src, DATA_COLUMNS].values
        fails = sum(
            1 for c in all_codes
            if c in shifted.index and c in anchor_idx.index and
            abs(float(shifted.loc[c, 'Storage Beginning of Month']) -
                float(anchor_idx.loc[c, 'Storage End of Month'])) > tol
        )
        return fails, shifted

    for n in list(range(1, max_shift + 1)) + list(range(-1, -max_shift - 1, -1)):
        fails, shifted = apply_shift(n)
        if fails == 0:
            return n, shifted
    return None, None


def repair_carry_swaps(data_csv, out_csv=None, tol=CARRY_REPAIR_TOL, dry_run=False):
    """
    Scan an extracted CSV for carry consistency failures and auto-repair them.

    Parameters
    ----------
    data_csv  : path to extracted/cleaned CSV (str or Path)
    out_csv   : output path; defaults to data_csv (in-place). Ignored in dry_run.
    tol       : carry tolerance in barrels (default 2)
    dry_run   : if True, print proposed changes without writing anything

    Returns (repaired_df, repair_log_df)
    """
    df = pd.read_csv(data_csv, dtype={'id': str})
    df['date'] = pd.to_datetime(df['date'], dayfirst=True)
    df['balanced'] = (
        df['balanced'].astype(str).str.upper()
        .map({'TRUE': True, 'FALSE': False, '1': True, '0': False})
    )
    out_csv = Path(out_csv or data_csv)

    repair_log = []
    n_carry_swaps = n_profile_swaps = n_shifts = n_unresolved = 0

    for fid, fac_df in df.groupby('id'):
        fac_df = fac_df.sort_values('date').copy()
        fully_bal = _fully_balanced_months(fac_df)

        for m_date in sorted(fac_df['date'].unique()):
            month_rows = fac_df[fac_df['date'] == m_date].copy()
            month_idx  = month_rows.set_index('Code')

            # ── Activity profile: rolling 24-month window before m_date ──
            cutoff     = m_date - pd.DateOffset(months=PROFILE_WINDOW)
            history_df = fac_df[(fac_df['date'] < m_date) & (fac_df['date'] >= cutoff)]
            profile    = _activity_profile(history_df)

            month_active = {int(r['Code']): float(r[DATA_COLUMNS].sum()) > 0
                            for _, r in month_rows.iterrows()}
            unexpected_active   = [c for c in range(1, 25)
                                    if month_active.get(c) and profile.get(c, 1.0) < INACTIVE_THRESHOLD]
            unexpected_inactive = [c for c in range(1, 25)
                                    if not month_active.get(c) and profile.get(c, 0.0) > ACTIVE_THRESHOLD]

            # ── Find nearest trustworthy balanced anchor ──
            anchor_idx, anchor_date = None, None
            for cand_date in sorted([d for d in fully_bal if d < m_date], reverse=True):
                cand_rows    = fac_df[fac_df['date'] == cand_date]
                anc_cutoff   = cand_date - pd.DateOffset(months=PROFILE_WINDOW)
                anc_hist     = fac_df[(fac_df['date'] < cand_date) & (fac_df['date'] >= anc_cutoff)]
                anc_profile  = _activity_profile(anc_hist)
                if _is_anchor_trustworthy(cand_rows, anc_profile):
                    anchor_idx, anchor_date = cand_rows.set_index('Code'), cand_date
                    break

            bad_codes = _carry_bad_codes(month_idx, anchor_idx, tol) if anchor_idx is not None else []

            if not bad_codes and not unexpected_active:
                continue   # nothing to repair

            if dry_run:
                print(f'  [{fid}] {m_date.strftime("%Y-%m")}: '
                      f'carry_fail={bad_codes}  '
                      f'unexpected_active={[PRODUCT_CODES.get(c, c) for c in unexpected_active]}')

            new_idx     = month_idx.copy()
            month_fixed = False

            # ── Layer 1: Carry-based pairwise swap ──────────────────────────
            if bad_codes and anchor_idx is not None:
                swaps = _find_pairwise_swaps(new_idx, anchor_idx, bad_codes, tol)
                if swaps:
                    new_idx = _apply_row_swaps(new_idx, swaps)
                    for a, b in swaps:
                        repair_log.append({'id': fid, 'date': m_date,
                            'anchor_date': anchor_date, 'action': 'swap_carry',
                            'code_a': a, 'product_a': PRODUCT_CODES.get(a, a),
                            'code_b': b, 'product_b': PRODUCT_CODES.get(b, b)})
                    n_carry_swaps += len(swaps)
                    if not _carry_bad_codes(new_idx, anchor_idx, tol):
                        month_fixed = True

            # ── Layer 2: Activity-profile anomaly swap ───────────────────────
            if unexpected_active and unexpected_inactive and not month_fixed:
                for ua in unexpected_active[:]:
                    for ui in unexpected_inactive[:]:
                        test = _apply_row_swaps(new_idx, [(ua, ui)])
                        test_ua_active = float(test.loc[ua, DATA_COLUMNS].sum()) > 0 if ua in test.index else False
                        test_ui_active = float(test.loc[ui, DATA_COLUMNS].sum()) > 0 if ui in test.index else False
                        if not test_ua_active and test_ui_active:
                            # Ensure the swap doesn't worsen carry (if anchor exists)
                            if anchor_idx is not None and bad_codes:
                                if len(_carry_bad_codes(test, anchor_idx, tol)) >= len(bad_codes):
                                    continue
                            new_idx = test
                            repair_log.append({'id': fid, 'date': m_date,
                                'anchor_date': anchor_date, 'action': 'swap_profile',
                                'code_a': ua, 'product_a': PRODUCT_CODES.get(ua, ua),
                                'code_b': ui, 'product_b': PRODUCT_CODES.get(ui, ui),
                                'profile_rate_a': round(profile.get(ua, 0.0), 3),
                                'profile_rate_b': round(profile.get(ui, 0.0), 3)})
                            n_profile_swaps += 1
                            month_fixed = True
                            break
                    if month_fixed:
                        break

            # ── Layer 3: Block shift ─────────────────────────────────────────
            if bad_codes and not month_fixed and anchor_idx is not None:
                shift_n, shifted_idx = _find_block_shift(new_idx, anchor_idx, tol)
                if shift_n is not None:
                    new_idx = shifted_idx
                    repair_log.append({'id': fid, 'date': m_date,
                        'anchor_date': anchor_date, 'action': 'block_shift',
                        'shift': shift_n})
                    n_shifts += 1
                    month_fixed = True

            # ── Still unresolved ─────────────────────────────────────────────
            if not month_fixed:
                repair_log.append({'id': fid, 'date': m_date,
                    'anchor_date': anchor_date, 'action': 'needs_review',
                    'bad_codes': str(bad_codes),
                    'unexpected_active': str([PRODUCT_CODES.get(c, c) for c in unexpected_active])})
                n_unresolved += 1
                continue

            # ── Write repaired values back to master df ──────────────────────
            if not dry_run:
                for code in range(1, 25):
                    if code in new_idx.index:
                        mask = (df['id'] == fid) & (df['date'] == m_date) & (df['Code'] == code)
                        for col in DATA_COLUMNS:
                            df.loc[mask, col] = new_idx.loc[code, col]

    # ── Summary ──────────────────────────────────────────────────────────────
    label = 'DRY RUN — nothing written' if dry_run else str(out_csv)
    print(f'Repair summary  ({label}):')
    print(f'  Carry swaps applied  : {n_carry_swaps}')
    print(f'  Profile swaps applied: {n_profile_swaps}')
    print(f'  Block shifts applied : {n_shifts}')
    print(f'  Still needs review   : {n_unresolved}')

    log_df = pd.DataFrame(repair_log)
    if not dry_run:
        df = validate_balance(df)   # recompute balance_diff / balanced after repairs
        _safe_csv_write(df, out_csv)
        if not log_df.empty:
            log_path = out_csv.parent / f'repair_log_{out_csv.stem}.csv'
            _safe_csv_write(log_df, log_path)
            print(f'Repair log       → {log_path}')

    return df, log_df


print('repair_carry_swaps() ready.')

repair_carry_swaps() ready.


---
## 2.C. Calling the fuctions: Parsing PDFs

In [18]:
# ── Run batch extraction ──
# Outputs go to  data/extracted/<id>.csv
# Logs go to     logs/extraction_log_<id>.csv
# Checkpoints:   logs/checkpoints/

# ── Configure run ──
facility  = ['10-0013']          # which refineries to process
date_from = '2025-11'            # skip PDFs before this month  (None = all years)
date_to   = None                 # skip PDFs after  this month  (None = no limit)

# ── Run ──
new_data, new_log = batch_extract(
    facility_filter=facility,
    date_from=date_from,
    date_to=date_to,
    use_claude=False,
    resume=False,   # uncomment to force reprocess from scratch
)

# ── Other examples ──
# Full run, all facilities, 2020 onwards:
# new_data, new_log = batch_extract(date_from='2020-01')

# All years for one facility:
# new_data, new_log = batch_extract(facility_filter=['01-0692'], date_from=None)

# Tesseract only (no API cost):
# new_data, new_log = batch_extract(use_claude=False)

# ── Post-process ──
if not new_data.empty:
    new_data['balance_diff'] = (
        new_data['Storage Beginning of Month'] + new_data['Receipts'] + new_data['Products Manufactured']
    ) - (
        new_data['Input'] + new_data['Fuel Used'] + new_data['Deliveries'] + new_data['Storage End of Month']
    )
    new_data['balanced'] = new_data['balance_diff'].abs() < 2
    new_data['material'] = new_data['Code'].map(PRODUCT_CODES)

    active = new_data[new_data[DATA_COLUMNS].sum(axis=1) > 0]
    print(f'Active rows: {len(active)}, Balanced: {active["balanced"].sum()} ({active["balanced"].mean():.0%})')
    print(f'Rows needing review: {(~active["balanced"]).sum()}')

# Save extracted data and log per facility
for fid in (facility or [new_data['id'].iloc[0]] if not new_data.empty else []):
    fd = new_data[new_data['id'] == fid] if 'id' in new_data.columns else new_data
    fl = new_log[new_log['id'] == fid]  if 'id' in new_log.columns  else new_log
    fd.to_csv(OUTPUT_DIR / f'{fid}.csv', index=False)
    fl.to_csv(LOG_DIR    / f'extraction_log_{fid}.csv', index=False)
    print(f'Saved {fid}: {len(fd)} rows → {OUTPUT_DIR}/{fid}.csv')

print(f'Total retried rows: {new_log["retried_rows"].sum() if not new_log.empty else 0}')

Date filter: skipping 135 PDFs outside 2025-11 – now
2 PDFs. Starting fresh.

BATCH COMPLETE: 2 processed, $0.00 API cost
Tiers: {1: np.int64(1), 3: np.int64(1)}
Modes: {'camelot': np.int64(1), 'scanned_ocr': np.int64(1)}
Saved to logs\checkpoints
Active rows: 16, Balanced: 0 (0%)
Rows needing review: 16
Saved 10-0013: 48 rows → data\extracted/10-0013.csv
Total retried rows: 0


#### Retry block

In [ ]:
# ── Run retry / OCR upgrade for a facility ──
# Set the facility you want to retry below, then run this cell.
#
# upgrade_ocr=True  → also re-runs scanned_ocr rows with pass_rate < 0.7 through Claude
#                     (rows already extracted by claude_vision are never touched — no duplicates)
# upgrade_ocr=False → only retries mode='failed' rows (original behaviour)
#
# After the retry completes, re-run the post-process cell above to refresh
# the per-facility CSV in data/extracted/.
#
# Close any open Excel files before running — they lock the CSVs.

retry_facility = '03-0006'   # ← change to the facility you want to retry

retry_log, retry_data = retry_failed(
    log_csv         = CHECKPOINT_DIR / 'extraction_log.csv',
    data_csv        = CHECKPOINT_DIR / 'extracted_rows.csv',
    facility_filter = [retry_facility],
    upgrade_ocr     = True,   # re-run low-quality OCR entries through Claude
)

# Inspect which ones still fail and why:
if retry_log is not None:
    still_failed = retry_log[(retry_log['mode'] == 'failed') & (retry_log['id'] == retry_facility)]
    if still_failed.empty:
        print(f'All previously-failed {retry_facility} PDFs now have data.')
    else:
        print(f'\n{len(still_failed)} still failing — Claude raw error:')
        cols = [c for c in ['file', 'error', 'claude_error', 'cost'] if c in still_failed.columns]
        print(still_failed[cols].to_string(index=False))

In [ ]:
# ── Post-process ──
if not retry_data.empty:
    retry_data['balance_diff'] = (
        retry_data['Storage Beginning of Month'] + retry_data['Receipts'] + retry_data['Products Manufactured']
    ) - (
        retry_data['Input'] + retry_data['Fuel Used'] + retry_data['Deliveries'] + retry_data['Storage End of Month']
    )
    retry_data['balanced'] = retry_data['balance_diff'].abs() < 2
    retry_data['material'] = retry_data['Code'].map(PRODUCT_CODES)

    active = retry_data[retry_data[DATA_COLUMNS].sum(axis=1) > 0]
    print(f'Active rows: {len(active)}, Balanced: {active["balanced"].sum()} ({active["balanced"].mean():.0%})')
    print(f'Rows needing review: {(~active["balanced"]).sum()}')

retry_data.to_csv(OUTPUT_DIR / f'{retry_facility}.csv', index=False)
retry_log.to_csv(LOG_DIR    / f'extraction_log_{retry_facility}.csv', index=False)

### Repair carry swaps 

In [ ]:
# ── Run carry repair ──────────────────────────────────────────────────────────
# Point at any extracted or cleaned CSV.
# dry_run=True first to preview changes; set to False to apply.

REPAIR_TARGET = CLEANED_DIR / f'{facility[0]}.csv'

# Preview what would change (no writes):
# repaired_df, repair_log = repair_carry_swaps(REPAIR_TARGET, dry_run=True)

# Apply repairs in-place once you are happy with the preview:
# repaired_df, repair_log = repair_carry_swaps(REPAIR_TARGET, dry_run=False)

# Optionally repair to a separate file:
repaired_df, repair_log = repair_carry_swaps(
    REPAIR_TARGET,
    out_csv=CLEANED_DIR / f'{facility[0]}_repaired.csv',
    dry_run=False,
)

if repair_log is not None and not repair_log.empty:
    print(f'\nRepair log ({len(repair_log)} entries):')
    display(repair_log.sort_values(['action', 'date']))

### Reconstruct log

In [ ]:

def reconstruct_log():
    import datetime
    rows_df = pd.read_csv(CHECKPOINT_DIR / 'extracted_rows.csv', dtype={'id': str})
    log_df  = pd.read_csv(CHECKPOINT_DIR / 'extraction_log.csv', dtype={'id': str})
    log_keys  = set(zip(log_df['id'], log_df['date']))
    rows_keys = set(zip(rows_df['id'], rows_df['date']))
    missing   = rows_keys - log_keys
    print(f"Missing from log: {len(missing)} entries")
    synth = []
    for (fid, date_str) in sorted(missing):
        dt   = datetime.datetime.strptime(date_str, '%Y-%m-%d')
        yr, mo = dt.year, dt.month
        matches = list(RAW_DATA_DIR.rglob(f"{yr}_{mo}___{fid}_*.pdf"))
        fp_str  = str(matches[0]) if matches else f"RECONSTRUCTED_{fid}_{date_str}.pdf"
        sub  = rows_df[(rows_df['id'] == fid) & (rows_df['date'] == date_str)]
        mode = sub['extraction_mode'].iloc[0] if not sub.empty else 'unknown'
        tier = int(sub['extraction_tier'].iloc[0]) if not sub.empty else 3
        active_mask = sub[DATA_COLUMNS].sum(axis=1) > 0
        active   = int(active_mask.sum())
        balanced = int(sub.loc[active_mask, 'balanced'].sum()) if active > 0 else 0
        pass_rate = round(balanced / active, 4) if active > 0 else 0.0
        synth.append({'file': fp_str, 'id': fid, 'date': date_str,
                    'mode': mode, 'tier': tier, 'error': '', 'claude_error': '',
                    'active': active, 'balanced': balanced, 'pass_rate': pass_rate,
                    'cost': 0.0, 'calls': 0, 'retried_rows': 0})
    if synth:
        new_log = pd.concat([log_df, pd.DataFrame(synth)], ignore_index=True)
        new_log.to_csv(CHECKPOINT_DIR / 'extraction_log.csv', index=False)
        print(f"Patched {len(synth)} rows → log now has {len(new_log)} entries")
    else:
        print("Log already complete, nothing to patch.")

---
## 3. Quick Test: Single PDFs

In [ ]:
def test_pdf(path, use_claude=True):
    if not os.path.exists(path): print(f'Not found: {path}'); return None
    r = extract_r1(path, use_claude=use_claude)
    print(f'File: {os.path.basename(path)}')
    print(f'Mode: {r["mode"]} | Tier: {r.get("tier")} | Header: {r["header"]}')
    if r.get('error'): print(f'Error: {r["error"]}')
    s = r.get('balance_stats',{})
    if s: print(f'Balance: {s.get("balanced_count",0)}/{s.get("total_active",0)} ({s.get("pass_rate",0):.0%})')
    a = r.get('api_usage',{})
    if a.get('calls',0)>0: print(f'API: {a["calls"]} call(s), ${a["cost_usd"]:.4f}')
    if not r['data']: return None
    df = validate_balance(pd.DataFrame(r['data']))
    df['material'] = df['Code'].map(PRODUCT_CODES)
    df['confidence'] = df['balanced'].map({True:'ok',False:'REVIEW'})
    pd.set_option('display.float_format','{:,.0f}'.format)
    print(df[['Code','material']+DATA_COLUMNS+['balance_diff','confidence']].to_string(index=False))
    return df

In [ ]:
# df = test_pdf('2025-june-03-0052.pdf')         # Digital (Blanchard)
# df = test_pdf('2026-january-03-0138.pdf')       # Clean scan (ExxonMobil)
# df = test_pdf('2025_1___02-0057_DIAMOND SHAMROCK REFINING COMPANY.pdf')  # Difficult scan
print('Test cell ready.')

---
## 4. Compare to Existing Database

In [ ]:
def compare(new_df, existing_path=EXISTING_DB_PATH):
    ex = pd.read_excel(existing_path, dtype={'id':str})
    ex['date'] = pd.to_datetime(ex['date'])
    nw = new_df.copy(); nw['date'] = pd.to_datetime(nw['date'])
    m = nw.merge(ex, on=['id','date','Code'], how='outer', suffixes=('_new','_old'), indicator=True)
    new_only = m[m['_merge']=='left_only']
    both = m[m['_merge']=='both']
    oc = [f'{c}_old' for c in DATA_COLUMNS if f'{c}_old' in both.columns]
    nc = [f'{c}_new' for c in DATA_COLUMNS if f'{c}_new' in both.columns]
    silent = both[(both[oc].fillna(0).sum(axis=1)==0)&(both[nc].fillna(0).sum(axis=1)>0)]
    print(f'Both: {len(both)} | Silent zeros: {len(silent)} | New only: {len(new_only)}')
    return {'silent_zeros':silent,'new_only':new_only}

# comp = compare(new_data)
print('Compare cell ready.')

---
## 5. Merge & Save

In [ ]:
def merge_db(new_df, existing_path=EXISTING_DB_PATH):
    ex = pd.read_excel(existing_path, dtype={'id':str})
    ex['date'] = pd.to_datetime(ex['date'])
    nw = new_df[['id','date','Code']+DATA_COLUMNS].copy()
    nw['date'] = pd.to_datetime(nw['date'])
    mg = ex.merge(nw, on=['id','date','Code'], how='outer', suffixes=('','_new'))
    for col in DATA_COLUMNS:
        nc = f'{col}_new'
        if nc in mg.columns:
            z=mg[col].fillna(0)==0; h=mg[nc].fillna(0)!=0
            mg.loc[z&h,col]=mg.loc[z&h,nc]
            mg[col]=mg[col].fillna(mg[nc])
            mg.drop(columns=[nc],inplace=True)
    return mg.sort_values(['id','date','Code']).reset_index(drop=True)

# final = merge_db(new_data)
# final.to_csv(OUTPUT_DIR / 'rrc_r1_database.csv', index=False)
print('Merge cell ready.')

---
## Notes

**Crash recovery:** Results save to `output/checkpoints/` after every PDF. If the loop fails:
1. Fix the issue (API key, network, etc.)
2. Re-run `batch_extract()` — it skips already-processed PDFs automatically
3. No duplicate API calls, no lost data

To start completely fresh: `batch_extract(resume=False)`

**Image size limit:** Images auto-compress to JPEG under 4.5MB before sending to Claude.

**Cost tracking:** Every result includes `api_usage.cost_usd` (0 for Tier 1/2). Batch summary shows total.

| Tier | Method | Accuracy | Cost |
|------|--------|----------|------|
| 1 | pdfplumber | ~100% | Free |
| 2 | Tesseract | 60-75% | Free |
| 3 | Claude vision + retry | ~95%+ | ~$0.01-0.05/page |

---
## 5. Build Database

Merges all cleaned per-refinery CSVs into a single flat file and then loads both production data and refinery metadata into a SQLite database.

**Outputs**
- `data/all_production.csv` — flat merge of all 16 cleaned CSVs (~27 K rows)
- `data/rrc_texas.db` — SQLite database with two tables:
  - `production` — all production records
  - `refineries` — refinery metadata from `reference/refinery_mapping.csv`

In [ ]:
import pathlib
import pandas as pd

CLEANED_DIR = pathlib.Path("data/cleaned")
OUTPUT_CSV  = pathlib.Path("data/all_production.csv")


f = Path('reference') / 'refinery_mapping.csv'

ref_names = pd.read_csv(f)

# The 15 canonical columns — files with extra trailing empty columns are trimmed to these.
CANONICAL_COLS = [
    "Code", "Storage Beginning of Month", "Receipts", "Input",
    "Products Manufactured", "Fuel Used", "Deliveries",
    "Storage End of Month", "id", "date", "extraction_mode",
    "extraction_tier", "balance_diff", "balanced", "material",
]

frames = []
for csv_path in sorted(CLEANED_DIR.glob("*.csv")):
    df = pd.read_csv(csv_path, dtype=str)
    # Drop any unnamed / empty trailing columns produced by variant files
    df = df[[c for c in df.columns if c in CANONICAL_COLS]]
    # Ensure column order is consistent
    df = df[CANONICAL_COLS]
    frames.append(df)

all_production = pd.concat(frames, ignore_index=True)
# Merging with refinery names and mappings
all_production = all_production.merge(ref_names, on = 'id')
# Saving final file to memory
all_production.to_csv(OUTPUT_CSV, index=False)

print(f"Merged {len(frames)} files → {len(all_production):,} rows")
print(f"Saved to: {OUTPUT_CSV.resolve()}")
all_production.head(3)

In [ ]:
f = Path('reference') / 'refinery_mapping.csv'

ref_names = pd.read_csv(f)



In [ ]:
all_production

In [ ]:
all_production.merge(ref_names, on = 'id')

In [ ]:
ref_names

In [ ]:
import sqlite3
import pathlib
import pandas as pd

OUTPUT_CSV   = pathlib.Path("data/all_production.csv")
MAPPING_PATH = pathlib.Path("reference/refinery_mapping.csv")
DB_PATH      = pathlib.Path("data/rrc_texas.db")

# ---------- production table ----------
production = pd.read_csv(OUTPUT_CSV, dtype=str)

# ---------- refineries table ----------
mapping_raw = pd.read_csv(MAPPING_PATH, dtype=str)

# Keep only the 8 named columns; the rest are unnamed trailing artifact columns
MAPPING_COLS = ["waterborne_map", "iir_map", "code", "ownser",
                "plant_name", "plant_location", "cleaned", "last_format"]
refineries = mapping_raw[[c for c in mapping_raw.columns if c in MAPPING_COLS]]
refineries = refineries[MAPPING_COLS]

# Drop rows where code is missing, blank, or clearly not an RRC code
# (catches the empty footer rows and the stray numeric rows at the bottom)
refineries = refineries[
    refineries["code"].notna() &
    (refineries["code"].str.strip() != "") &
    (~refineries["code"].str.fullmatch(r"[\d\.]+").fillna(False))  # reject pure numeric cells
]
refineries = refineries.reset_index(drop=True)

# Rename the typo column to keep things clean
refineries = refineries.rename(columns={"ownser": "owner"})

# ---------- write to SQLite ----------
if DB_PATH.exists():
    DB_PATH.unlink()  # start fresh each run

conn = sqlite3.connect(DB_PATH)
production.to_sql("production", conn, index=False, if_exists="replace")
refineries.to_sql("refineries", conn, index=False, if_exists="replace")
conn.close()

print(f"production : {len(production):,} rows")
print(f"refineries : {len(refineries):,} rows")
print(f"Saved to   : {DB_PATH.resolve()}")

# Quick sanity-check join
conn = sqlite3.connect(DB_PATH)
sample = pd.read_sql("""
    SELECT p.date, r.plant_name, p.material, p.[Products Manufactured]
    FROM production p
    JOIN refineries r ON p.id = r.code
    WHERE p.material = 'Motor Gasoline'
    LIMIT 5
""", conn)
conn.close()
sample